In [1]:
# Re-install the unsloth environment
!pip install --no-cache-dir --upgrade unsloth unsloth_zoo trl peft loralib sentencepiece

In [3]:
from unsloth import FastLanguageModel
import torch

# 1. Load the base model (as you did in Phase 1)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-4-e2b", 
    max_seq_length = 512, # Use 512 for DPO to save memory
    dtype = None,
    load_in_4bit = True,
)

# 2. Add LoRA adapters (Must use the same configuration as Phase 2!)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

# 3. Load your SAVED adapter weights
# If you pushed to HF:
# Load the adapter with a defined name (e.g., "bible_adapter")
model.load_adapter("r-karra/gemma-4-bible-sft", adapter_name="bible_adapter")

# If you downloaded them locally to your Kaggle storage:
# model.load_adapter("/kaggle/input/your-dataset-folder/final_bible_aligned_adapter")

==((====))==  Unsloth 2026.6.9: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/101M [00:00<?, ?B/s]

<All keys matched successfully>

In [5]:
import os
print("Files in /kaggle/working/:")
print(os.listdir("/kaggle/working/"))

Files in /kaggle/working/:
['.virtual_documents', 'unsloth_compiled_cache']


In [6]:
# 1. Save the model weights to the disk
model.save_pretrained("/kaggle/working/final_bible_aligned_adapter")
tokenizer.save_pretrained("/kaggle/working/final_bible_aligned_adapter")

# 2. Verify they are now saved
print("Files now in /kaggle/working/:")
print(os.listdir("/kaggle/working/"))

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/final_bible_aligned_adapter/tokenizer_config.json.


Files now in /kaggle/working/:
['.virtual_documents', 'final_bible_aligned_adapter', 'unsloth_compiled_cache']


In [7]:
import shutil
# Remove large temporary caches before saving
shutil.rmtree("/kaggle/working/unsloth_compiled_cache", ignore_errors=True)

# Check what is currently in your working folder
print(os.listdir("/kaggle/working/"))

['.virtual_documents', 'final_bible_aligned_adapter']


In [9]:
# Assuming you have already run model.save_pretrained(...)
# 1. Login to HF if you haven't yet (use your secret)
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)

# 2. Push the adapter directly to the Hub
# Replace 'your-username' with your actual HF username
model.push_to_hub("r-karra/gemma-4-bible-sft", token=hf_token)
tokenizer.push_to_hub("r-karra/gemma-4-bible-sft", token=hf_token)

print("✅ Model uploaded safely to Hugging Face Hub!")

README.md:   0%|          | 0.00/563 [00:00<?, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Saved model to https://huggingface.co/r-karra/gemma-4-bible-sft


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpeqw3xeh2/tokenizer_config.json.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Model uploaded safely to Hugging Face Hub!
